# VICReg Pretrain -> SupCon Fine-Tune

Two-stage embedding pipeline:

1. **(Already done) VICReg pretraining** — task-agnostic; learned a
   decorrelated, full-rank starting embedding from masked / unmasked views.
   The checkpoint lives at `VICREG_CKPT`.
2. **(This notebook) SupCon fine-tune** — load the pretrained encoder,
   attach a fresh SupCon projection head, and fine-tune end-to-end with the
   target label. We use a **two-phase** schedule:
   - *Phase 1 (head warm-up):* encoder frozen + in `eval()` mode; only the
     SupCon projector trains, so large early head gradients can't damage the
     pretrained encoder weights.
   - *Phase 2 (fine-tune):* unfreeze; train both with **discriminative LRs**
     (small LR for the pretrained encoder, larger LR for the fresh head).

Per-epoch we report mean train SupCon loss, holdout SupCon loss (no grad),
and holdout linear-probe **AUC + PR-AUC** (PR-AUC has more headroom when the
target is imbalanced and ROC-AUC saturates). Best epoch by holdout AUC is
saved to `runs/supcon_finetune/best.pt`.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from ts_embed.data import (DatasetPaths, TimeSeriesDataset,
                           TimeFeatureMasker, ContrastiveCollator)
from ts_embed.model import TSEncoder, TSEncoderConfig, TSEmbeddingModel
from ts_embed.loss import SupConLoss, VICRegLoss, VICRegConfig

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Configuration

Point these at your data directories and your VICReg checkpoint. The
checkpoint must be a `torch.save` dict containing `model` (a
`TSEmbeddingModel` or bare `TSEncoder` state_dict) and `cfg` (the
`TSEncoderConfig.__dict__`). The training scripts and the starter notebook
save in this format.

In [ ]:
# >>> point these at your data and your VICReg checkpoint <<<
DATA_DIR    = REPO_ROOT / 'data_demo'
TRAIN_DIR   = DATA_DIR / 'train'
HOLDOUT_DIR = DATA_DIR / 'holdout'
VICREG_CKPT = REPO_ROOT / 'runs' / 'vicreg' / 'pretrain.pt'

# Fine-tuning hyperparameters.
BATCH_SIZE           = 256
ENCODER_LR           = 2e-4   # small LR for the pretrained encoder
HEAD_LR              = 2e-3   # larger LR for the fresh SupCon projector
WEIGHT_DECAY         = 1e-4
HEAD_WARMUP_EPOCHS   = 2       # phase 1: encoder frozen, projector only
FINETUNE_EPOCHS      = 13      # phase 2: encoder unfrozen, discriminative LR
TEMPERATURE          = 0.1
print('config loaded')

## 2. (Demo only) Bootstrap data + a VICReg checkpoint

**Skip this cell** if you already have the train/holdout files and a VICReg
checkpoint at the paths above. Otherwise it generates synthetic data and runs
a short VICReg pretrain so the notebook runs end-to-end out of the box.

In [ ]:
N_DEMO, T, F_NUM, F_CAT = 6000, 24, 98, 2

def _files_exist(d):
    return all((d / f).exists() for f in
               ('numeric.npy', 'missing.npy', 'categorical.npy', 'target.npy'))

if not (_files_exist(TRAIN_DIR) and _files_exist(HOLDOUT_DIR)):
    print('generating synthetic train / holdout demo data ...')
    rng = np.random.default_rng(0)
    f_delinq  = rng.standard_normal(N_DEMO).astype(np.float32)
    f_payment = rng.standard_normal(N_DEMO).astype(np.float32)
    f_balance = rng.standard_normal(N_DEMO).astype(np.float32)
    numeric = (0.3 * rng.standard_normal((N_DEMO, T, F_NUM))).astype(np.float32)
    t_axis = np.linspace(0, 1, T, dtype=np.float32)[None, :, None]
    numeric[:, :, 0:33]  += f_balance[:, None, None] * (0.5 + t_axis)
    numeric[:, :, 33:66] += f_payment[:, None, None] + 0.3 * rng.standard_normal(
        (N_DEMO, T, 33)).astype(np.float32)
    numeric[:, :, 66:98] += f_delinq[:, None, None] * t_axis * 1.5
    missing = (rng.random((N_DEMO, T, F_NUM)) < 0.1).astype(np.uint8)
    feat_mean = numeric.mean((0, 1), keepdims=True)
    numeric = np.where(missing == 1, feat_mean, numeric).astype(np.float32)
    categorical = np.stack([
        (rng.random((N_DEMO, T)) < 0.5).astype(np.int8),
        (rng.random((N_DEMO, T)) < 0.5).astype(np.int8),
    ], axis=-1)
    logit = 1.3 * f_delinq - 1.0 * f_payment + 0.6 * f_balance - 1.4
    prob = 1.0 / (1.0 + np.exp(-logit))
    target = (rng.random(N_DEMO) < prob).astype(np.int64)

    perm = rng.permutation(N_DEMO)
    train_rows   = perm[:int(0.85 * N_DEMO)]
    holdout_rows = perm[int(0.85 * N_DEMO):]

    def _save(d, rows):
        d.mkdir(parents=True, exist_ok=True)
        np.save(d / 'numeric.npy',     numeric[rows])
        np.save(d / 'missing.npy',     missing[rows])
        np.save(d / 'categorical.npy', categorical[rows])
        np.save(d / 'target.npy',      target[rows])
    _save(TRAIN_DIR,   train_rows)
    _save(HOLDOUT_DIR, holdout_rows)
    print('  wrote train / holdout to', DATA_DIR)
else:
    print('train / holdout files already exist, skipping data gen')

if not VICREG_CKPT.exists():
    print('no VICReg checkpoint at', VICREG_CKPT, '- running brief demo pretrain ...')
    pre_paths = DatasetPaths(numeric=TRAIN_DIR / 'numeric.npy',
                              missing=TRAIN_DIR / 'missing.npy',
                              categorical=TRAIN_DIR / 'categorical.npy')
    pre_ds = TimeSeriesDataset(pre_paths)
    pre_masker = TimeFeatureMasker(time_mask_prob=0.25, feature_mask_prob=0.30,
                                   n_time_spans=2, feat_span_frac=0.5)
    pre_loader = DataLoader(pre_ds, batch_size=256, shuffle=True, drop_last=True,
                            num_workers=0, collate_fn=ContrastiveCollator(pre_masker))
    pre_cfg = TSEncoderConfig(n_numeric=F_NUM, cat_cardinalities=(2, 2), seq_len=T,
                              d_model=128, n_layers=3, n_heads=4, proj_dim=128)
    pre_model = TSEmbeddingModel(pre_cfg).to(device)
    vicreg = VICRegLoss(VICRegConfig(gather_distributed=False))
    pre_opt = torch.optim.AdamW(pre_model.parameters(), lr=1e-3, weight_decay=1e-4)
    pre_model.train()
    for ep in range(5):
        for batch in pre_loader:
            num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
            keep_a = batch['time_keep_a'].to(device)
            num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
            keep_b = batch['time_keep_b'].to(device)
            cat_a = batch['categorical_a'].to(device); cat_b = batch['categorical_b'].to(device)
            _, z_a = pre_model(num_a, mis_a, cat_a, keep_a)
            _, z_b = pre_model(num_b, mis_b, cat_b, keep_b)
            loss = vicreg(z_a, z_b)['loss']
            pre_opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(pre_model.parameters(), 1.0)
            pre_opt.step()
        print(f'  vicreg epoch {ep}: loss = {float(loss):.3f}')
    VICREG_CKPT.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'model': pre_model.state_dict(), 'cfg': pre_cfg.__dict__}, VICREG_CKPT)
    print('  saved demo checkpoint ->', VICREG_CKPT)
    del pre_model, pre_opt, pre_loader, pre_ds, vicreg, pre_masker
    if device.type == 'cuda':
        torch.cuda.empty_cache()
else:
    print('VICReg checkpoint already exists, skipping demo pretrain')

## 3. Load the VICReg-pretrained encoder

Rebuild a `TSEncoder` from the saved `cfg` and load just the encoder weights
from the checkpoint. The projector weights (used only by VICReg) are
discarded — SupCon will train its own.

In [ ]:
ckpt = torch.load(VICREG_CKPT, map_location='cpu', weights_only=False)
assert 'cfg' in ckpt and 'model' in ckpt, (
    "checkpoint must contain `cfg` and `model`; got keys: " + str(list(ckpt)))

cfg = TSEncoderConfig(**ckpt['cfg'])
encoder = TSEncoder(cfg).to(device)

# Strip DDP / torch.compile prefixes if present, then keep only encoder.* keys.
state = {k.removeprefix('_orig_mod.').removeprefix('module.'): v
         for k, v in ckpt['model'].items()}
enc_state = {k[len('encoder.'):]: v for k, v in state.items()
             if k.startswith('encoder.')}
if not enc_state:           # bare TSEncoder checkpoint
    enc_state = {k: v for k, v in state.items() if not k.startswith('projector.')}
incompatible = encoder.load_state_dict(enc_state, strict=True)
print('VICReg encoder loaded; incompatible keys:', incompatible)
print('  cfg:', cfg)

## 4. Datasets, SupCon loss, and the discriminative-LR optimizer

Each item carries its target (no global-index bookkeeping). A
`WeightedRandomSampler` balances batches so the supervised positives are
meaningful even with an imbalanced target. The optimizer holds **two param
groups** from the start — encoder (small LR) and SupCon projector (larger LR)
— so we can freeze/unfreeze via `requires_grad` without rebuilding it.

In [ ]:
train_paths = DatasetPaths(
    numeric=TRAIN_DIR / 'numeric.npy',
    missing=TRAIN_DIR / 'missing.npy',
    categorical=TRAIN_DIR / 'categorical.npy')
holdout_paths = DatasetPaths(
    numeric=HOLDOUT_DIR / 'numeric.npy',
    missing=HOLDOUT_DIR / 'missing.npy',
    categorical=HOLDOUT_DIR / 'categorical.npy')

train_base   = TimeSeriesDataset(train_paths)
holdout_base = TimeSeriesDataset(holdout_paths)
target_train   = torch.from_numpy(np.load(TRAIN_DIR   / 'target.npy')).long()
target_holdout = torch.from_numpy(np.load(HOLDOUT_DIR / 'target.npy')).long()


class TargetDataset(Dataset):
    """Wraps a base dataset so each item yields its target alongside the
    feature dict."""
    def __init__(self, base, targets):
        assert len(base) == len(targets), 'base and targets length mismatch'
        self.base = base; self.targets = targets
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        item = self.base[i]
        item['target'] = self.targets[i]
        return item


train_ds   = TargetDataset(train_base,   target_train)
holdout_ds = TargetDataset(holdout_base, target_holdout)

masker = TimeFeatureMasker(time_mask_prob=0.25, feature_mask_prob=0.30,
                           n_time_spans=2, feat_span_frac=0.5)
_cc = ContrastiveCollator(masker)

def collate(batch):
    targets = torch.stack([b.pop('target') for b in batch])
    out = _cc(batch)
    out['target'] = targets
    return out

cls_count = torch.bincount(target_train, minlength=2).float()
sample_w = (1.0 / cls_count)[target_train]
sampler = WeightedRandomSampler(sample_w, num_samples=len(train_ds), replacement=True)

train_loader   = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                            drop_last=True, num_workers=0, collate_fn=collate)
holdout_loader = DataLoader(holdout_ds, batch_size=BATCH_SIZE, shuffle=False,
                            drop_last=False, num_workers=0, collate_fn=collate)

supcon = SupConLoss(in_dim=cfg.d_model, proj_dim=128, proj_hidden=256,
                    temperature=TEMPERATURE).to(device)

opt = torch.optim.AdamW([
    {'params': encoder.parameters(), 'lr': ENCODER_LR, 'weight_decay': WEIGHT_DECAY},
    {'params': supcon.parameters(),  'lr': HEAD_LR,    'weight_decay': WEIGHT_DECAY},
])
print(f'train n={len(train_ds)} pos rate={float(target_train.float().mean()):.3f} | '
      f'holdout n={len(holdout_ds)} pos rate={float(target_holdout.float().mean()):.3f}')

## 5. Two-phase fine-tune

Per-epoch metrics:
- `train_loss` -- mean SupCon loss across train batches.
- `holdout_loss` -- SupCon loss on the holdout in `eval()` mode (the masker
  is reseeded so the loss is reproducible across epochs).
- `AUC` -- holdout linear-probe ROC-AUC.
- `PR-AUC` -- holdout linear-probe average precision. Use this for selection
  when ROC-AUC saturates (common with imbalanced targets).

In [ ]:
def roc_auc(y, s):
    y = np.asarray(y); s = np.asarray(s)
    try:
        from sklearn.metrics import roc_auc_score
        return float(roc_auc_score(y, s))
    except ImportError:
        order = np.argsort(s, kind='mergesort')
        ranks = np.empty(len(s), float)
        sr = s[order]; i = 0
        while i < len(sr):
            j = i
            while j + 1 < len(sr) and sr[j + 1] == sr[i]:
                j += 1
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0
            i = j + 1
        pos = y == 1
        npos = int(pos.sum()); nneg = len(y) - npos
        if npos == 0 or nneg == 0:
            return float('nan')
        return float((ranks[pos].sum() - npos * (npos + 1) / 2) / (npos * nneg))

def pr_auc(y, s):
    """Average precision = area under the precision-recall curve."""
    y = np.asarray(y); s = np.asarray(s)
    try:
        from sklearn.metrics import average_precision_score
        return float(average_precision_score(y, s))
    except ImportError:
        order = np.argsort(-s, kind='mergesort')
        ys = y[order]
        tp = np.cumsum(ys); fp = np.cumsum(1.0 - ys)
        precision = tp / np.maximum(tp + fp, 1.0)
        recall = tp / max(int(y.sum()), 1)
        recall_prev = np.concatenate([[0.0], recall[:-1]])
        return float(np.sum((recall - recall_prev) * precision))

@torch.no_grad()
def embed_dataset(ts_dataset):
    encoder.eval()
    n_rows = int(len(ts_dataset))
    Z = []
    for start in range(0, n_rows, 512):
        end = start + 512 if start + 512 < n_rows else n_rows
        sl = slice(start, end)
        num = torch.from_numpy(np.asarray(ts_dataset.numeric[sl], dtype=np.float32)).to(device)
        mis = torch.from_numpy(np.asarray(ts_dataset.missing[sl], dtype=np.float32)).to(device)
        cat = torch.from_numpy(np.asarray(ts_dataset.categorical[sl], dtype=np.int64)).to(device)
        Z.append(encoder(num, mis, cat).float().cpu().numpy())
    return np.concatenate(Z)

def probe_metrics():
    Ztr = embed_dataset(train_base); Zho = embed_dataset(holdout_base)
    try:
        from sklearn.linear_model import LogisticRegression
        lp = LogisticRegression(max_iter=2000, class_weight='balanced')
        lp.fit(Ztr, target_train.numpy())
        proba = lp.predict_proba(Zho)[:, 1]
        return roc_auc(target_holdout.numpy(), proba), pr_auc(target_holdout.numpy(), proba)
    except ImportError:
        return float('nan'), float('nan')

@torch.no_grad()
def holdout_loss():
    encoder.eval(); supcon.eval()
    rng_state = torch.get_rng_state(); torch.manual_seed(123)
    try:
        total, n_seen = 0.0, 0
        for batch in holdout_loader:
            num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
            keep_a = batch['time_keep_a'].to(device)
            num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
            keep_b = batch['time_keep_b'].to(device)
            cat_a = batch['categorical_a'].to(device)
            cat_b = batch['categorical_b'].to(device)
            tgt = batch['target'].to(device)
            ea = encoder(num_a, mis_a, cat_a, keep_a)
            eb = encoder(num_b, mis_b, cat_b, keep_b)
            l = supcon(ea, eb, tgt)['loss']
            bs = num_a.size(0)
            total += float(l) * bs
            n_seen += bs
    finally:
        torch.set_rng_state(rng_state)
    return total / max(n_seen, 1)

run_dir = REPO_ROOT / 'runs' / 'supcon_finetune'
run_dir.mkdir(parents=True, exist_ok=True)
best_auc, best_state = -float('inf'), None
history = []

for epoch in range(HEAD_WARMUP_EPOCHS + FINETUNE_EPOCHS):
    finetuning = epoch >= HEAD_WARMUP_EPOCHS
    for p in encoder.parameters():
        p.requires_grad = finetuning
    encoder.train(finetuning)   # eval mode during warm-up: no dropout in features
    supcon.train()

    train_total, train_n = 0.0, 0
    for batch in train_loader:
        num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
        keep_a = batch['time_keep_a'].to(device)
        num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
        keep_b = batch['time_keep_b'].to(device)
        cat_a = batch['categorical_a'].to(device)
        cat_b = batch['categorical_b'].to(device)
        tgt = batch['target'].to(device)

        emb_a = encoder(num_a, mis_a, cat_a, keep_a)
        emb_b = encoder(num_b, mis_b, cat_b, keep_b)
        loss = supcon(emb_a, emb_b, tgt)['loss']

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(supcon.parameters()), 1.0)
        opt.step()
        bs = num_a.size(0)
        train_total += float(loss) * bs
        train_n += bs

    train_loss = train_total / max(train_n, 1)
    h_loss = holdout_loss()
    auc, prauc = probe_metrics()
    history.append({'epoch': epoch, 'phase': 'finetune' if finetuning else 'head-warmup',
                    'train_loss': train_loss, 'holdout_loss': h_loss,
                    'auc': auc, 'pr_auc': prauc})

    improved = auc > best_auc
    if improved:
        best_auc = auc
        best_state = {
            'encoder': {k: v.detach().cpu().clone() for k, v in encoder.state_dict().items()},
            'supcon':  {k: v.detach().cpu().clone() for k, v in supcon.state_dict().items()},
            'cfg': cfg.__dict__, 'epoch': epoch,
            'val_auc': auc, 'val_pr_auc': prauc,
        }

    tag = 'finetune   ' if finetuning else 'head-warmup'
    print(f'epoch {epoch:2d} [{tag}]  train={train_loss:.3f}  '
          f'holdout={h_loss:.3f}  AUC={auc:.4f}  PR-AUC={prauc:.4f}'
          + ('  <- best' if improved else ''))

torch.save(best_state, run_dir / 'best.pt')
print(f'\nbest val AUC = {best_auc:.4f}  (PR-AUC={best_state["val_pr_auc"]:.4f})  '
      f'@ epoch {best_state["epoch"]}  ->  {run_dir / "best.pt"}')

## 6. Evaluate the best fine-tuned embedding

Reload the best epoch's encoder weights and report both linear-probe and kNN
ROC-AUC / PR-AUC plus a collapse guard.

In [ ]:
encoder.load_state_dict(best_state['encoder'])

Ztr = embed_dataset(train_base); Zte = embed_dataset(holdout_base)
ytr = target_train.numpy(); yte = target_holdout.numpy()

# 1. linear probe
try:
    from sklearn.linear_model import LogisticRegression
    lp = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Ztr, ytr)
    probas = lp.predict_proba(Zte)[:, 1]
    lin_auc, lin_prauc = roc_auc(yte, probas), pr_auc(yte, probas)
except ImportError:
    lin_auc = lin_prauc = float('nan')

# 2. cosine kNN (sensitive to local geometry; sometimes moves when AUC saturates)
def knn_score(Xtr, ytr, Xte, k=25):
    Xtr = Xtr / (np.linalg.norm(Xtr, axis=1, keepdims=True) + 1e-8)
    Xte = Xte / (np.linalg.norm(Xte, axis=1, keepdims=True) + 1e-8)
    sims = Xte @ Xtr.T
    nn = np.argpartition(-sims, k, axis=1)[:, :k]
    return ytr[nn].mean(axis=1)
knn_s = knn_score(Ztr, ytr, Zte)
knn_auc, knn_prauc = roc_auc(yte, knn_s), pr_auc(yte, knn_s)

print(f'best-epoch linear probe : AUC={lin_auc:.4f}  PR-AUC={lin_prauc:.4f}')
print(f'best-epoch kNN          : AUC={knn_auc:.4f}  PR-AUC={knn_prauc:.4f}')

std = Ztr.std(0)
print(f'embedding per-dim std: mean={std.mean():.3f} min={std.min():.3f}')
assert std.mean() > 1e-2, 'COLLAPSE: embedding nearly constant'

mu_pos = Ztr[ytr == 1].mean(0); mu_neg = Ztr[ytr == 0].mean(0)
between = np.linalg.norm(mu_pos - mu_neg)
within = np.linalg.norm(Ztr - np.where(ytr[:, None] == 1, mu_pos, mu_neg), axis=1).mean()
print(f'class-mean separation ratio = {between / within:.2f}')

## Next steps

- **Downstream model**: load `best.pt`, rebuild the encoder, and feed `Z` to
  your XGBoost / GBM — ideally **concatenated** with your engineered features,
  not replacing them:
  ```python
  ckpt = torch.load(run_dir / 'best.pt', map_location='cpu', weights_only=False)
  cfg = TSEncoderConfig(**ckpt['cfg'])
  enc = TSEncoder(cfg).to(device)
  enc.load_state_dict(ckpt['encoder']); enc.eval()
  ```
- **Model selection**: if ROC-AUC saturates, switch the `improved` check to
  `prauc > best_pr_auc` so the best epoch is picked by PR-AUC instead.
- **Layer-wise LR decay**: for very long fine-tunes, set encoder LR per layer
  (deeper layers move more, shallow layers stay near the pretrained weights).
- **Light masking only**: if fine-tuning destabilizes, reduce `time_mask_prob`
  / `feature_mask_prob` for the SupCon stage — the encoder is already
  pretrained and may not need as much augmentation noise.
- **Avoid label leakage**: if you'll evaluate downstream on rows different
  from `target_holdout`, make sure SupCon was trained *only* on rows that
  appear in the downstream training fold.